# 41 · 候选缓存、TTL与一致性

单元：S11。先修：40、26–28。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：逻辑时钟和构造库存；无真实缓存流量。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 缓存对象
特征缓存存向量，候选缓存存doc IDs，结果缓存含最终排序。此处只实现候选缓存。命中省loader调用，统计的是模拟成本而非毫秒收益。

In [2]:
cache=CandidateCache(ttl=5)
for t in [0,1,2,6]: print(t,cache.get('apple',t,0,lambda:['a','b']))
assert cache.cost==2
print('loader calls',cache.cost)

0 (['a', 'b'], 'miss')
1 (['a', 'b'], 'hit')
2 (['a', 'b'], 'hit')
6 (['a', 'b'], 'miss')
loader calls 2


## 陈旧缓存
删除a、generation变更但忽略版本，TTL未到仍命中。只改变check_generation。

In [3]:
stale=CandidateCache(); stale.get('apple',0,0,lambda:['a','b'])
bad,state=stale.get('apple',1,1,lambda:['b'],check_generation=False)
good,state2=stale.get('apple',1,1,lambda:['b'])
print('stale',bad,state,'version checked',good,state2)
assert 'a' in bad and 'a' not in good

stale ['a', 'b'] hit version checked ['b'] miss


## 过期与故障回退
空候选可缓存，但短TTL防止持续漏召。后端故障应显式回退词法，不能把错误当空结果。

In [4]:
empty=CandidateCache(ttl=1)
print(empty.get('rare',0,0,lambda:[]))
print(empty.get('rare',2,0,lambda:['d']))
trace=pipeline('phone repair',fail_semantic=True)
assert trace['fallback']=='lexical' and trace['ranking']
print(trace)

([], 'miss')
(['d'], 'miss')
{'query': 'phone repair', 'intent': 'support', 'lexical': ['c', 'd', 'a', 'b'], 'semantic': [], 'cache': 'miss', 'candidates': ['a', 'b', 'c', 'd'], 'ranking': ['c', 'd', 'a', 'b'], 'fallback': 'lexical', 'empty': False}


## 成本与差异
固定请求序列，只改TTL；较长TTL提高命中但无generation检查会滞后。无真实延迟测量，不推线上收益。

In [5]:
for ttl in [1,10]:
    c=CandidateCache(ttl); results=[]
    for t in range(4): results.append(c.get('q',t,0,lambda:[str(t)])[0])
    print(ttl,'calls',c.cost,'results',results)

1 calls 4 results [['0'], ['1'], ['2'], ['3']]
10 calls 1 results [['0'], ['0'], ['0'], ['0']]


## 本次保存结果的解读

四次访问在TTL=5时只调用loader两次；忽略generation返回已删除a，启用版本检查立即刷新。TTL=1每次重载，TTL=10只重载一次但持续返回旧值；这是命中与新鲜度的构造取舍，并非本机真实服务加速测量。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。